<a href="https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = on post corresponding to a particular client on a particular date

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q datasets huggingface_hub pandas pyarrow
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

import pandas as pd

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df = pd.read_parquet(url)

df.info()
df.head()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9841378 entries, 0 to 9841377
Data columns (total 31 columns):
 #   Column                    Dtype   
---  ------                    -----   
 0   report_date               object  
 1   client_hash_id            object  
 2   content_hash_id           object  
 3   client_has_gsc            bool    
 4   client_has_ga4            bool    
 5   gsc_data_available        bool    
 6   ga4_data_available        object  
 7   gsc_impressions           int64   
 8   gsc_clicks                int64   
 9   gsc_sum_position          int64   
 10  gsc_avg_position          float64 
 11  ga4_pageviews             float64 
 12  ga4_sessions              float64 
 13  ga4_users                 float64 
 14  ga4_engaged_sessions      float64 
 15  ga4_total_engagement_sec  float64 
 16  sessions_organic          float64 
 17  sessions_direct           float64 
 18  sessions_referral         float64 
 19  sessions_social           float64 
 20  se

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 2. Fields: feature / label / context / excluded

## Features

The features used to identify content refresh opportunities are:

- gsc_impressions: measures search visibility before the decision moment.
- gsc_clicks: measures the number of clicks received from search.
- gsc_avg_position: represents the average search ranking position.
- ga4_sessions: measures user visits to the content.
- ga4_engaged_sessions: measures meaningful user engagement.
- sessions_ai: measures traffic coming from AI sources.
- scroll_events: measures user interaction with the content.


## Label

- refresh_opportunity: a proxy label indicating whether a content item should be prioritized for refresh based on observed performance signals.


## Context

The following fields provide identification and temporal context:

- report_date: date of the observation.
- month: partition month of the data.
- client_hash_id: identifies the client.
- content_hash_id: identifies the content item.


## Excluded

The following fields are excluded:

- client_hash_id and content_hash_id: excluded as model features because they are identifiers and could make the model memorize specific entities instead of learning general patterns.

- Future performance information: excluded because it would not be available at the moment of making the decision and would cause data leakage.

In [28]:

# 2. Imputar posición
df["gsc_avg_position"] = df["gsc_avg_position"].fillna(
    df["gsc_avg_position"].mean()
)


# 3. Imputar GA4
ga4_features = [
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_ai",
    "scroll_events"
]

df[ga4_features] = df[ga4_features].fillna(0)


features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_ai",
    "scroll_events"
]


# 5. Dataset de entrada
X = df[features]

X

df["needs_refreshing"] = (
    (df["gsc_impressions"] > 10) &
    (df["gsc_clicks"] == 0)
).astype(int)

target = df['needs_refreshing']

df_corr = X.copy()

df_corr["needs_refreshing"] = df["needs_refreshing"]

df["needs_refreshing"].value_counts()

correlation = df_corr.corr(
    numeric_only=True
)

correlation


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,sessions_ai,scroll_events,needs_refreshing
gsc_impressions,1.000000,0.596687,-0.064770,0.213158,0.210776,0.061990,0.160932,0.162092
gsc_clicks,0.596687,1.000000,-0.071909,0.255270,0.299255,0.030780,0.221947,-0.048441
gsc_avg_position,-0.064770,-0.071909,1.000000,0.007515,-0.007636,0.006384,0.004008,-0.092916
ga4_sessions,0.213158,0.255270,0.007515,1.000000,0.211555,0.051467,0.747485,0.056478
ga4_engaged_sessions,0.210776,0.299255,-0.007636,0.211555,1.000000,0.087403,0.286515,0.006378
sessions_ai,0.061990,0.030780,0.006384,0.051467,0.087403,1.000000,0.042071,0.014513
scroll_events,0.160932,0.221947,0.004008,0.747485,0.286515,0.042071,1.000000,0.042818
needs_refreshing,0.162092,-0.048441,-0.092916,0.056478,0.006378,0.014513,0.042818,1.000000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
duplicated = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print(df.shape[0])
print(df['report_date'].max())
print(df['report_date'].min())



9841378
2026-03-31
2026-03-01


## Data observations and limitations

### Client distribution

The dataset contains observations from multiple clients, but the number of records is not evenly distributed. Some clients contribute a very large number of rows, while others have significantly fewer observations. This indicates that the historical data is concentrated around a subset of clients, so model performance may be influenced by clients with more available records.

### Search visibility distribution (GSC)

Search visibility is highly concentrated. Most content observations have zero impressions (`gsc_impressions = 0`), with 6,230,317 records having no observed impressions or clicks. Only a smaller portion of content receives search exposure, and a few content items account for very high impression values.

This means that zero impressions should not automatically be interpreted as poor content quality. It may indicate lack of search exposure, indexing limitations, or low demand.

### Click behavior

Clicks are also highly sparse. Many observations have impressions but no clicks, meaning that some content is visible in search results but does not generate user visits. These cases are more relevant for identifying refresh opportunities because the content already has search exposure.

### GA4 availability and engagement

GA4 sessions are mostly concentrated around zero values. The majority of observations have no recorded GA4 sessions, while only a smaller number of records show user engagement. This indicates that analytics availability and traffic levels vary significantly across observations.

### Target distribution

The proxy target `needs_refreshing` is imbalanced:

- Class 0: 8,163,936 observations
- Class 1: 1,677,442 observations

Most observations are classified as not needing refresh. Therefore, accuracy alone would not be a reliable evaluation metric, and metrics focused on identifying positive cases (such as recall) are more appropriate.

### Limitations

This dataset provides observed performance signals, but it cannot explain the reason behind user behavior. For example, low clicks may be caused by poor titles, search intent mismatch, competition, or other external factors that are not measured in the dataset.

Additionally, daily observations may contain repeated measurements of the same content over time, meaning that consecutive rows may not be completely independent.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(df['client_hash_id'].value_counts().head(60))
df.columns


print(df['report_date'].value_counts())
print(df['needs_refreshing'].value_counts())


print(df['gsc_impressions'].value_counts())
print(df['ga4_sessions'].value_counts())


print(df.groupby("gsc_impressions")["gsc_clicks"].sum())

print(df.groupby("gsc_impressions")["gsc_clicks"].size())
#

client_hash_id
client_625b6439094e23e4    988497
client_3ffa76342f366962    904847
client_73cda7b4e4f265ea    869640
client_08a6a72ff48e62c0    851275
client_62f4a7e64f5e0096    756660
client_65de48885f4ef01b    426307
client_23a62021009f63c4    423613
client_ba65e80a1116ae41    410409
client_2b4306c3ed003f01    375906
client_fef1a8f436438636    335379
client_3197e6291363b4db    316610
client_e547b89c05043229    288548
client_a80fca3f171ed1de    235505
client_19b89ee4fe3db6da    213001
client_f623b01661d4bfe4    182218
client_2094c6eb080311d5    173700
client_3f0ce4d44fe94f3d    170957
client_a60a11451483af1c    169167
client_cd12bcfd98942aa1    159948
client_ff644d8251367cbb    140182
client_20259bd6705d81d4    136720
client_157ffe4d4a595515    133632
client_b10cb2997d0c7c86    126512
client_4a18d1793d92fb84    122142
client_400c21c81c8b46ef    106808
client_9958f0a7ae1df715     95356
client_e5c2aa26a8598242     94253
client_2910fd937f0b4d9a     83119
client_795153d5b7850ccf     78709

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.